In [0]:
%pip install -r ../requirements.txt

In [0]:
catalog  = ""
schema   = ""
n_orders = 50_000
sp       = ""  # App service principal application (client) ID

In [0]:
import io, os, re, runpy, sys

sys.dont_write_bytecode = True  # prevent __pycache__ errors on serverless

_HERE = "/Workspace" + os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)

def run_script(script_name, **params):
    """Run a Python script in-process, forwarding non-empty kwargs as CLI --key=value args.
    Returns a dict of KEY=VALUE lines printed by the script (for chaining between steps)."""
    script = os.path.join(_HERE, script_name)
    sys.argv = [script] + [f"--{k.replace('_', '-')}={v}" for k, v in params.items() if v != ""]

    _buf, _orig = io.StringIO(), sys.stdout

    class _Tee:
        def write(self, s):  _orig.write(s); _buf.write(s)
        def flush(self):     _orig.flush()
        def __getattr__(self, n): return getattr(_orig, n)

    sys.stdout = _Tee()
    try:
        runpy.run_path(script, run_name="__main__")
    finally:
        sys.stdout = _orig

    # Parse KEY=VALUE lines (e.g. GENIE_SPACE_ID=abc123) from the script output
    outputs = {}
    for line in _buf.getvalue().splitlines():
        m = re.match(r'^([A-Z][A-Z0-9_]+)=(.+)', line.strip())
        if m:
            outputs[m.group(1)] = m.group(2)
    return outputs

In [0]:
run_script("01_generate_data.py", catalog=catalog, schema=schema, n_orders=n_orders)

In [0]:
run_script("02_generate_ka_documents.py", catalog=catalog, schema=schema)

In [0]:
_out03 = run_script("03_create_genie_space.py", catalog=catalog, schema=schema)


In [0]:
# Auto-captured from step 03 output — or paste GENIE_SPACE_ID manually
genie_space_id = globals().get("_out03", {}).get("GENIE_SPACE_ID") or ""

print(genie_space_id)

In [0]:
_out04 = run_script("04_create_knowledge_assistant.py", catalog=catalog, schema=schema)

In [0]:
# Auto-captured from step 04 output — or paste KA_ID / KA_ENDPOINT manually
ka_id       = globals().get("_out04", {}).get("KA_ID")       or ""
ka_endpoint = globals().get("_out04", {}).get("KA_ENDPOINT") or ""

print(ka_id)
print(ka_endpoint)

In [0]:
_out05 = run_script("05_create_supervisor_agent.py",
                    genie_space_id=genie_space_id,
                    ka_id=ka_id,
                    ka_endpoint=ka_endpoint)

In [0]:
# Auto-captured from step 05 output — or paste MAS_ENDPOINT manually
mas_endpoint = globals().get("_out05", {}).get("MAS_ENDPOINT") or ""

print(mas_endpoint)

## Before running step 06 — fill in `sp` in the Parameters cell

Step 06 grants permissions to the **Databricks App's service principal (SP)**. You need its **Application (client) ID** — a UUID that looks like `a1b2c3d4-...`.

### If you haven't created the App yet
1. In the left sidebar go to **Apps**
2. Click **Create app**
3. Give it a name (e.g. `panino-bricks`), choose a compute type, and click **Create**
4. Wait for the app to reach **Running** state

### Find the Application ID
1. Open the app's detail page (click its name in the Apps list)
2. In the **Settings** tab look for **Service principal** → copy the **Application ID**

   *Alternatively*: go to **Settings → Identity & Access → Service principals**, search for the app name, open it and copy the **Application ID** from the overview page.

### Set the value
Go back to **cell 2 (Parameters)** at the top and set:
```python
sp = "<paste Application ID here>"
```
Then re-run cell 2 and continue from cell 12.

In [0]:
run_script("06_grant_app_permissions.py",
           catalog=catalog, schema=schema, sp=sp,
           genie_space_id=genie_space_id,
           mas_endpoint=mas_endpoint,
           ka_endpoint=ka_endpoint)

In [0]:
run_script("07_grant_lakebase_role.py", sp=sp)